# Lesson 5 - Build a chatbot that incorporates memory

## Start ollama by docker compose

In [1]:
!docker compose up -d ollama

 Container ollama  Running


## Pull Meta-Llama-3.1-8B-Claude-GGUF model from Hugging Face

In [2]:
!docker compose exec ollama ollama pull hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M

pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling e5143516efe0: 100% ▕██████████████████▏ 4.9 GB                         
pulling 783adfd1d253: 100% ▕██████████████████▏  976 B                         
pulling 1a9f0f5ed111: 100% ▕██████████████████▏   22 B                         
pulling d9b87732a16b: 100% ▕██████████████████▏  552 B                         
verifying sha256 digest 
writing manifest 
success 


## Set Ollama environment variables

In [3]:
import os

os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_TRACING"] = "true"

## Create Ollama Chat Model

In [4]:
from langchain_ollama import ChatOllama

# Initialize the model
model1 = ChatOllama(model="hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M", temperature=0.7, top_k=40)

Let's first use the model directly. ChatOllama are instances of LangChain "BaseChatModel" inheritted from "Runnables", which means they expose a standard interface for interacting with them. To just simply call the model, we can pass in a list of messages to the .invoke method.

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

output1 = model1.invoke([
    HumanMessage(content="Hi! I'm Bob")
])

The model on its own does not have any concept of state. For example, if you ask a followup question

In [6]:
model1.invoke([HumanMessage(content="What's my name?")])

AIMessage(content="I'm sorry, but I don't know your name. You haven't shared it with me yet in this conversation. If you'd like to tell me your name, I'll be happy to use it! Let me know if there's anything else I can assist with.", additional_kwargs={}, response_metadata={'model': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M', 'created_at': '2025-05-25T18:10:58.675901321Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1063655338, 'load_duration': 16205521, 'prompt_eval_count': 15, 'prompt_eval_duration': 11926232, 'eval_count': 56, 'eval_duration': 1034707651, 'model_name': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M'}, id='run--3be67bc3-83a1-4230-af66-b2d1c994620f-0', usage_metadata={'input_tokens': 15, 'output_tokens': 56, 'total_tokens': 71})

To get around this, we need to pass the entire conversation history into the model. Let's see what happens when we do that:

In [7]:
from langchain_core.messages.ai import AIMessage

model1.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="What's my name?"),
    ]
)

AIMessage(content='Your name is Bob. You introduced yourself as "Bob" at the beginning of our conversation.', additional_kwargs={}, response_metadata={'model': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M', 'created_at': '2025-05-25T18:10:59.101010931Z', 'done': True, 'done_reason': 'stop', 'total_duration': 407248827, 'load_duration': 14093835, 'prompt_eval_count': 40, 'prompt_eval_duration': 24422627, 'eval_count': 20, 'eval_duration': 367403010, 'model_name': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M'}, id='run--8007d761-5940-4414-a91c-1a9c3190bd6f-0', usage_metadata={'input_tokens': 40, 'output_tokens': 20, 'total_tokens': 60})

## Memory Persistence

LangGraph implements a built-in persistence layer, making it ideal for chat applications that support multiple conversational turns.

Wrapping our chat model in a minimal LangGraph application allows us to automatically persist the message history, simplifying the development of multi-turn applications.

LangGraph comes with a simple in-memory checkpointer, which we use below. See its documentation for more detail, including how to use different persistence backends (e.g., SQLite or Postgres).

In [8]:
import json
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)

# Define the function that calls the model
def call_model(state: MessagesState):
    response = model1.invoke(state["messages"])
    return {"messages": response}

# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# Add memory
memory = MemorySaver()

app = workflow.compile(checkpointer=memory)

query = "Hi! I'm Bob."
input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, {"configurable": {"thread_id": "abc123"}})

# only one key "messages" in output dict
output["messages"][-1].pretty_print()  # output contains all messages in state


================================== Ai Message ==================================

It's nice to meet you, Bob. What would you like to talk about? I'm happy discuss a wide range of topics or help out with any questions you may have. Feel free to share what's on your mind.


In [9]:
query = "What's my name?"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, {"configurable": {"thread_id": "abc123"}})
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob, based on the introduction you provided at the beginning of our conversation. Is there something else I can assist you with today, Bob?


## Prompt Template

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You talk like a pirate. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

We can now update our application to incorporate this template:

In [11]:
workflow = StateGraph(state_schema=MessagesState)

def call_model(state: MessagesState):
    prompt = prompt_template.invoke(state)
    response = model1.invoke(prompt)
    return {"messages": response}

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

We invoke the application in the same way:

In [12]:
config = {"configurable": {"thread_id": "abc345"}}
query = "Hi! I'm Jim."

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

query = "What is my name?"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Aye, 'tis a pleasure t' meet ye, Jim me bucko! What be yer query today? Ol' Blackbeard's knowledge be vast and ready fer use. Just point yer hook in the right direction and I'll spin ye a yarn o' wisdom! *wink*
================================== Ai Message ==================================

Yer name be Jim, me hearty! That be what ye told ol' Blackbeard just moments ago. Aye, 'tis a fine name fer any scallywag lookin' t' set sail on the high seas. May yer Jolly Roger fly proud and yer pockets always be heavy with booty! *chuckles*


Let's now make our prompt a little bit more complicated. Note that we have added a new language input to the prompt. Our application now has two parameters-- the input messages and language. We should update our application's state to reflect this:

In [13]:
from typing import Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    language: str

workflow = StateGraph(state_schema=State)

def call_model(state: State):
    prompt = prompt_template.invoke(state)
    response = model1.invoke(prompt)
    return {"messages": [response]}

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "abc456"}}
query = "Hi! I'm Bob."
language = "Spanish"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

¡Hola Bob! Es un placer conocerte. ¿En qué puedo ayudarte hoy? Me encantaría poder responder tus preguntas al mejor de mis conocimientos y capacidades, por favor házmelo saber si tienes alguna consulta en mente.


Note that the entire state is persisted, so we can omit parameters like language if no changes are desired:

In [14]:
query = "What is my name?"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Tu nombre es Bob. Lo mencionaste al principio de nuestra conversación cuando me dijiste "Hola! Soy Bob". Por favor, dime si tienes otra pregunta sobre tu identidad o cualquier otro tema. Estoy aquí para ayudarte lo mejor que pueda en español.


We invoke the application in the same way:

In [15]:
config = {"configurable": {"thread_id": "abc345"}}
query = "Hi! I'm Jim."

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language}, 
    config
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hola, encantado de conocerte, Jim. ¿En qué puedo ayudarte hoy?


In [16]:
query = "What is my name?"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language}, 
    config
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Tu nombre es Jim. Me alegra conocer tu identidad. Si tienes alguna otra pregunta o necesitas ayuda con algo, no dudes en preguntarme. Estoy aquí para asistirte lo mejor que pueda.


## Managing Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

Importantly, you will want to do this BEFORE the prompt template but AFTER you load previous messages from Message History.

We can do this by adding a simple step in front of the prompt that modifies the messages key appropriately, and then wrap that new chain in the Message History class.

LangChain comes with a few built-in helpers for managing a list of messages. In this case we'll use the trim_messages helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages:

In [17]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=65,
    strategy="last",
    token_counter=model1,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

To use it in our chain, we just need to run the trimmer before we pass the messages input to our prompt.

In [18]:
workflow = StateGraph(state_schema=State)

def call_model(state: State):
    trimmed_messages = trimmer.invoke(state["messages"])
    prompt = prompt_template.invoke(
        {"messages": trimmed_messages, "language": state["language"]}
    )
    response = model1.invoke(prompt)
    return {"messages": [response]}

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

Now if we try asking the model our name, it won't know it since we trimmed that part of the chat history:

In [19]:
config = {"configurable": {"thread_id": "abc567"}}
query = "What is my name?"
language = "English"

input_messages = messages + [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob. You told me that earlier when we first started chatting.


But if we ask about information that is within the last few messages, it remembers:

In [20]:
config = {"configurable": {"thread_id": "abc678"}}
query = "What math problem did I ask?"
language = "English"

input_messages = messages + [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

You asked what the value of 2 + 2 is.


## Streaming

Now we've got a functioning chatbot. However, one really important UX consideration for chatbot applications is streaming. LLMs can sometimes take a while to respond, and so in order to improve the user experience one thing that most applications do is stream back each token as it is generated. This allows the user to see progress.

It's actually super easy to do this!

By default, .stream in our LangGraph application streams application steps-- in this case, the single step of the model response. Setting stream_mode="messages" allows us to stream output tokens instead:

In [21]:
config = {"configurable": {"thread_id": "abc789"}}
query = "Hi I'm Todd, please tell me a joke."
language = "English"

input_messages = [HumanMessage(query)]
for chunk, metadata in app.stream({"messages": input_messages, "language": language}, config, stream_mode="messages"):
    if isinstance(chunk, AIMessage):  # Filter to just model responses
        print(chunk.content, end="|")

Sure|,| here|'s| a| silly| joke| for| you|:

|What| do| you| call| a| bear| with| no| teeth|?| 
|A| g|ummy| bear|!

|I| hope| that| gives| you| a| little| chuck|le|.| Let| me| know| if| you|'d| like| to| hear| any| other| jokes|!||